# Fine-tuning Qwen2.5-0.5B for ATIS structured-output — **Unsloth** version

This is a drop-in counterpart to `atis_finetune_colab.ipynb`: **same data, schema,
hyperparameters, and evaluation**, but the model load + LoRA + training use
[Unsloth](https://github.com/unslothai/unsloth) instead of plain transformers/TRL/PEFT.
Run both and compare speed, memory, and quality.

**Schema** (8 keys, `null` when not mentioned):
```json
{"intent": ["flight"], "from_city": null, "to_city": null,
 "depart_date": null, "depart_time": null, "airline": null,
 "round_trip": null, "class_type": null}
```

All randomness is seeded (`SEED`) so the split and training are reproducible across runs.

> **Runtime:** set `Runtime -> Change runtime type -> GPU` (a T4 is enough). Unsloth requires a GPU.

## 1. Install dependencies (Colab uses pip)

In [ ]:
!pip install -q unsloth
# If you hit a torchao version error, run `!pip install -U torchao` and restart the runtime.

## 2. Config + schema

In [ ]:
from unsloth import FastLanguageModel  # import before transformers so Unsloth can patch it
import json, urllib.request
import torch
from transformers import set_seed

SEED = 42
set_seed(SEED)  # seed python / numpy / torch for reproducible runs

MODEL_NAME = "unsloth/Qwen2.5-0.5B-Instruct"  # Unsloth-optimized mirror of the same weights
MAX_SEQ_LENGTH = 1024
TEST_FRACTION = 0.10

INTENT_ORDER = ["flight", "airfare", "ground_service", "airline"]
CHOSEN = set(INTENT_ORDER)
SCHEMA_FIELDS = ["from_city", "to_city", "depart_date", "depart_time",
                 "airline", "round_trip", "class_type"]
FIELD_MAP = {
    "fromloc": "from_city", "toloc": "to_city",
    "depart_date": "depart_date", "depart_time": "depart_time",
    "airline_name": "airline", "airline_code": "airline",
    "round_trip": "round_trip", "class_type": "class_type",
}

SYSTEM_PROMPT = (
    "You extract structured information from airline-travel queries. "
    "Respond with ONLY a JSON object with exactly these keys: "
    'intent, from_city, to_city, depart_date, depart_time, airline, round_trip, class_type. '
    "'intent' is a list whose values are a subset of "
    '["flight", "airfare", "ground_service", "airline"]. '
    "Every other field is a string copied from the query, or null if not mentioned. "
    "Output only the JSON, no extra text."
)

## 3. Download ATIS, build JSON records, and make a fixed 90/10 split

Identical transform to the vanilla notebook: pool ATIS's predefined train/test files and make a
single **seeded** 90/10 split. All spans of a field are joined so multi-token values (e.g.
"june 12") stay whole.

In [ ]:
BASE = "https://huggingface.co/datasets/tuetschek/atis/resolve/main"
for split in ["train", "test"]:
    urllib.request.urlretrieve(f"{BASE}/atis_{split}.csv", f"atis_{split}.csv")

import csv, random

def read_csv(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))

def normalize_intent(intent_str):
    parts = set(intent_str.split("+"))
    if not parts <= CHOSEN:
        return None
    return [i for i in INTENT_ORDER if i in parts]

def extract_spans(text, slots):
    spans, base, cur = [], None, []
    for tok, tag in zip(text.split(), slots.split()):
        if tag == "O":
            if base is not None:
                spans.append((base, " ".join(cur))); base, cur = None, []
            continue
        prefix, b = tag[0], tag[2:]
        if prefix == "B" or b != base:
            if base is not None:
                spans.append((base, " ".join(cur)))
            base, cur = b, [tok]
        else:
            cur.append(tok)
    if base is not None:
        spans.append((base, " ".join(cur)))
    return spans

def to_record(text, slots):
    # Collect all spans per field (in order) and join them, so multi-token values
    # like "june 12" or "after 6 pm" are kept whole instead of only the first span.
    acc = {f: [] for f in SCHEMA_FIELDS}
    for b, span_text in extract_spans(text, slots):
        field = FIELD_MAP.get(b.split(".")[0])
        if field is not None:
            acc[field].append(span_text)
    return {f: " ".join(v) if v else None for f, v in acc.items()}

def build(path):
    rows = []
    for r in read_csv(path):
        intent = normalize_intent(r["intent"])
        if intent is None:
            continue
        rows.append({"query": r["text"],
                     "output": {"intent": intent, **to_record(r["text"], r["slots"])}})
    return rows

# Pool both predefined splits, then make a fixed seeded 90/10 split.
all_rows = build("atis_train.csv") + build("atis_test.csv")
random.Random(SEED).shuffle(all_rows)
n_test = round(len(all_rows) * TEST_FRACTION)
test_rows = all_rows[:n_test]
train_rows = all_rows[n_test:]
print(f"total: {len(all_rows)} | train: {len(train_rows)} | test: {len(test_rows)} "
      f"({len(test_rows)/len(all_rows):.1%} test)")
print("example:", json.dumps(train_rows[0], indent=2))

## 4. Format as prompt/completion pairs

Conversational **prompt** (system + user) and **completion** (assistant JSON). TRL applies the
chat template and trains on the completion only (the prompt is masked out of the loss).

In [ ]:
from datasets import Dataset

def to_prompt(query):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": query}]

def dump(record):
    return json.dumps(record, ensure_ascii=False)

def to_example(row):
    return {"prompt": to_prompt(row["query"]),
            "completion": [{"role": "assistant", "content": dump(row["output"])}]}

train_ds = Dataset.from_list([to_example(r) for r in train_rows])
test_ds = Dataset.from_list([to_example(r) for r in test_rows])
print(train_ds)
print("\ncompletion[0]:", train_ds[0]["completion"][0]["content"])

## 5. Load model + tokenizer + LoRA (Unsloth)

This is the main difference from the vanilla notebook: `FastLanguageModel` loads the model with
Unsloth's optimized kernels, and `get_peft_model` attaches the LoRA adapters (same rank/alpha).

In [ ]:
# Native bf16 only on Ampere+ (capability >= 8); a T4 (7.5) uses fp16. This must match the
# dtype Unsloth picks below so SFTConfig(bf16=...) is consistent.
bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,            # auto: bf16 if natively supported, else fp16
    load_in_4bit=False,    # match the vanilla notebook (16-bit LoRA, not QLoRA)
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Same LoRA config as the vanilla run (all-linear == these 7 projection modules for Qwen2).
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
print(f"loaded {MODEL_NAME} | bf16={bf16}")

## 6. Generation + evaluation helpers (identical to the vanilla notebook)

In [ ]:
@torch.no_grad()
def generate(queries, max_new_tokens=128, batch_size=16):
    """Greedy-decode JSON strings for a list of queries (batched, left-padded)."""
    model.eval()
    prev_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    out = []
    for i in range(0, len(queries), batch_size):
        batch = queries[i:i + batch_size]
        prompts = [tokenizer.apply_chat_template(to_prompt(q), tokenize=False,
                                                 add_generation_prompt=True) for q in batch]
        enc = tokenizer(prompts, return_tensors="pt", padding=True,
                        truncation=True, max_length=384).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
        for j in range(len(batch)):
            new = gen[j][enc["input_ids"].shape[1]:]
            out.append(tokenizer.decode(new, skip_special_tokens=True))
    tokenizer.padding_side = prev_side
    return out

def parse_json(text):
    s, e = text.find("{"), text.rfind("}")
    if s == -1 or e == -1 or e < s:
        return None
    try:
        return json.loads(text[s:e + 1])
    except json.JSONDecodeError:
        return None

def _norm(v):
    return v.strip().lower() if isinstance(v, str) else v

def evaluate(rows, max_new_tokens=128):
    preds = generate([r["query"] for r in rows], max_new_tokens=max_new_tokens)
    n = len(rows)
    valid = exact = intent_ok = 0
    field_ok = {f: 0 for f in SCHEMA_FIELDS}
    for row, text in zip(rows, preds):
        pred = parse_json(text)
        if pred is None:
            continue
        valid += 1
        gold = row["output"]
        gi = set(gold["intent"])
        pi = set(pred["intent"]) if isinstance(pred.get("intent"), list) else set()
        all_ok = gi == pi
        if all_ok:
            intent_ok += 1
        for f in SCHEMA_FIELDS:
            if _norm(pred.get(f)) == _norm(gold[f]):
                field_ok[f] += 1
            else:
                all_ok = False
        exact += all_ok
    print(f"n={n}")
    print(f"valid JSON   : {valid/n:6.1%}")
    print(f"exact match  : {exact/n:6.1%}")
    print(f"intent acc   : {intent_ok/n:6.1%}")
    for f in SCHEMA_FIELDS:
        print(f"  {f:<12}: {field_ok[f]/n:6.1%}")

def show(rows, idxs):
    preds = generate([rows[i]["query"] for i in idxs])
    for i, text in zip(idxs, preds):
        print("Q   :", rows[i]["query"])
        print("gold:", dump(rows[i]["output"]))
        print("pred:", text.strip())
        print("-" * 70)

## 7. Baseline (before fine-tuning)

In [ ]:
FastLanguageModel.for_inference(model)  # enable Unsloth's fast inference path
DEMO = [0, 5, 17, 42]
print("=== BEFORE: sample generations ===")
show(test_rows, DEMO)
print("\n=== BEFORE: metrics on the full test set ===")
evaluate(test_rows)

## 8. Fine-tune with LoRA (Unsloth + TRL `SFTTrainer`)

In [ ]:
from trl import SFTConfig, SFTTrainer

FastLanguageModel.for_training(model)  # switch back to training kernels
tokenizer.padding_side = "right"        # training

sft_config = SFTConfig(
    output_dir="atis-qwen-unsloth-lora",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=25,
    max_length=384,
    packing=False,
    bf16=bf16, fp16=not bf16,
    report_to="none",
    save_strategy="no",
    seed=SEED,
    data_seed=SEED,
)

# Note: no peft_config here — LoRA was already attached via get_peft_model in Section 5.
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    processing_class=tokenizer,
)
trainer.train()

## 9. After fine-tuning

In [ ]:
FastLanguageModel.for_inference(model)
print("=== AFTER: same sample generations ===")
show(test_rows, DEMO)
print("\n=== AFTER: metrics on the full test set ===")
evaluate(test_rows)

## 10. Save the adapter
Saves the LoRA adapter (small).

In [ ]:
model.save_pretrained("atis-qwen-unsloth-lora")
tokenizer.save_pretrained("atis-qwen-unsloth-lora")
print("saved adapter to ./atis-qwen-unsloth-lora")

## 11. (Optional) Export to GGUF for Ollama

Unsloth merges the adapter into 16-bit weights (`save_pretrained_merged`); we then convert to
GGUF with llama.cpp (same path as the vanilla notebook) and bundle a `Modelfile` for download.

In [ ]:
# Merge LoRA into 16-bit weights with Unsloth.
model.save_pretrained_merged("atis-qwen-merged", tokenizer, save_method="merged_16bit")

# Convert the merged model to GGUF with llama.cpp.
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip install -q gguf sentencepiece protobuf
!python llama.cpp/convert_hf_to_gguf.py atis-qwen-merged \
    --outfile atis-qwen-unsloth-f16.gguf --outtype f16

# Write an Ollama Modelfile using the exact training system prompt + Qwen's ChatML template.
template = """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
"""
modelfile = f'''FROM ./atis-qwen-unsloth-f16.gguf

TEMPLATE """{template}"""

SYSTEM """{SYSTEM_PROMPT}"""

PARAMETER temperature 0
PARAMETER stop "<|im_end|>"
'''
with open("Modelfile", "w") as f:
    f.write(modelfile)
print(modelfile)

# Bundle the GGUF + Modelfile and download.
!zip -q atis-qwen-unsloth-ollama.zip atis-qwen-unsloth-f16.gguf Modelfile
from google.colab import files; files.download("atis-qwen-unsloth-ollama.zip")

## 12. Run it in Ollama (on your machine)

Download the finished `atis-qwen-unsloth-ollama.zip`, unzip it, then from that folder:

```bash
ollama create atis-qwen-unsloth -f Modelfile
ollama run atis-qwen-unsloth "i need a flight tomorrow from columbus to minneapolis"
```

Using a distinct model name (`atis-qwen-unsloth`) lets you keep both versions in Ollama and
compare their outputs side by side.

## Comparing against the vanilla notebook
Run `atis_finetune_colab.ipynb` and this one on the same GPU and compare:
- **Speed** — wall-clock of the training cell (TRL logs runtime; or time it).
- **Memory** — peak VRAM (`torch.cuda.max_memory_allocated()/1e9`).
- **Quality** — the after-training metrics block (valid JSON / exact match / per-field).

Same data + seed + hyperparameters, so differences are down to Unsloth's kernels.